# [性能分析]{.lang-zh} [Profiling]{.lang-en}

```{admonition} 概要
:class: tip lang-zh
用 PETSc 的 `-log_view` 分析 Firedrake 程序的性能, 由日志数据生成火焰图, 用自定义 `Stage` 把程序分段统计, 并做一次强扩展性测试.
```

```{admonition} Overview
:class: tip lang-en
Profiling Firedrake programs with PETSc `-log_view`, generating flame graphs from the log data, dividing a program into custom stages, and running a strong scaling test.
```

::::lang-zh
Firedrake 程序的时间往往不是花在自己写的那几行代码里, 而是花在网格生成、
代码生成与编译、矩阵组装和线性方程组求解这些底层步骤上. Firedrake 建立在 PETSc
之上, 而 PETSc 自带一套性能日志: 它会记录每个内部步骤的调用次数、耗时和浮点运算数,
我们也可以用 `PETSc.Log.Event` 把自己关心的代码块标注成事件, 一并统计.

参考资料:

1. [Optimising Firedrake performance](https://www.firedrakeproject.org/optimising.html):
   Firedrake 官方的性能优化指南, 讲火焰图的生成与解读、自定义事件的写法以及几个常见的性能问题, 是本章的主要依据.
2. [PETSc: Profiling](https://petsc.org/main/manual/profiling/): PETSc 手册的性能分析一章. `-log_view` 的各种输出格式、输出表格每一列的含义都在这里.
3. [`PetscInitialize`](https://petsc.org/main/manualpages/Sys/PetscInitialize/):
   PETSc 初始化阶段识别的命令行选项一览, 各个 `-log_*` 选项都列在其中.
4. [`PetscLogView`](https://petsc.org/main/manualpages/Profiling/PetscLogView/):
   把日志写到指定 viewer 的接口. 本章在 Notebook 中打印完整表格用的
   `PETSc.Log.view` 就是它的 Python 版本.
::::

::::lang-en
The time a Firedrake program takes is usually not spent in the few lines one writes
oneself, but in the underlying steps: mesh generation, code generation and
compilation, matrix assembly and the solution of linear systems. Firedrake is
built on PETSc, and PETSc comes with a performance log: it records the number of
calls, the elapsed time and the number of floating-point operations of every
internal step, and we can also mark the blocks of code we care about as events
with `PETSc.Log.Event` so that they are counted as well.

References:

1. [Optimising Firedrake performance](https://www.firedrakeproject.org/optimising.html):
   Firedrake's official guide to performance optimization. It covers generating
   and reading flame graphs, how to write custom events and a few common
   performance problems; it is the main source for this chapter.
2. [PETSc: Profiling](https://petsc.org/main/manual/profiling/): the profiling
   chapter of the PETSc manual. The output formats of `-log_view` and the meaning
   of every column of the output tables are described here.
3. [`PetscInitialize`](https://petsc.org/main/manualpages/Sys/PetscInitialize/):
   a list of the command-line options recognized during PETSc initialization; all
   the `-log_*` options are among them.
4. [`PetscLogView`](https://petsc.org/main/manualpages/Profiling/PetscLogView/):
   the interface that writes the log to a given viewer. `PETSc.Log.view`, used in
   this chapter to print the full table in the notebook, is its Python version.
::::

## `log_view`

::::lang-zh
`-log_view` 是 PETSc 的命令行选项, 加在脚本后面即可, 不需要改动代码:
::::

::::lang-en
`-log_view` is a PETSc command-line option; it is appended after the script name on the command line and requires no change to the code:
::::

```bash
python3 your_script.py -log_view
```

::::lang-zh
程序结束时 PETSc 会把整个运行过程的性能摘要打印到标准输出. 冒号后面还可以指定输出文件和格式, 常用的几种是:

| 选项 | 作用 |
| ---- | ---- |
| `-log_view` | 把性能摘要打印到标准输出, 最常用 |
| `-log_view :profile.txt` | 同上, 但写入文件 `profile.txt` |
| `-log_view :profile.txt:ascii_flamegraph` | 输出火焰图格式, 见下面的"火焰图"一节 |
| `-log_view :profile.xml:ascii_xml` | 输出为按调用关系嵌套的 XML |
| `-log_view_memory` | 在事件表中额外给出内存分配的列 |

此外还有 `-log_all` (每个进程各写一个 `Log.rank` 文件)、`-log_trace` (实时打印每一步在做什么) 等选项, 完整列表见
[PETSc 手册的 Profiling 一章](https://petsc.org/main/manual/profiling/).
::::

::::lang-en
When the program finishes, PETSc prints a performance summary of the whole run to
standard output. An output file and a format can also be given after the colon;
the common ones are:

| Option | Effect |
| ------ | ------ |
| `-log_view` | print the performance summary to standard output, the most common one |
| `-log_view :profile.txt` | the same, but written to the file `profile.txt` |
| `-log_view :profile.txt:ascii_flamegraph` | output in flame graph format, see the section "Flame graphs" below |
| `-log_view :profile.xml:ascii_xml` | output as XML nested according to the call structure |
| `-log_view_memory` | add columns for memory allocation to the event table |

There are further options such as `-log_all` (each process writes its own
`Log.rank` file) and `-log_trace` (print what every step is doing in real time);
for the full list see
[the Profiling chapter of the PETSc manual](https://petsc.org/main/manual/profiling/).
::::

### [读懂 `-log_view` 的输出]{.lang-zh} [Reading the output of `-log_view`]{.lang-en}

::::lang-zh
`-log_view` 的输出分三段.

第一段是全局摘要, 给出整个运行过程的墙钟时间 `Time (sec)`、浮点运算总数 `Flops`、
MPI 消息条数和全局归约次数等. 每行有 `Max`、`Max/Min`、`Avg`、`Total` 四列, 分别是各进程上的最大值、最大值与最小值之比、平均值和总和. 其中 `Max/Min` 接近 1 说明各进程负载均衡, 这个比值越大负载越不均衡; 串行运行时非零量的比值恒为 1 (某一项本身为 0 时 PETSc 直接打成 `0.0`), 没有参考价值.

第二段 `Summary of Stages` 按 `Stage` 汇总. 不特意划分 `Stage` 时, 所有内容都记在
`Main Stage` 里.

第三段是事件表, 每个事件占一行, 既包括 `KSPSolve`、`MatMult` 这样的 PETSc 内部事件,
也包括自己用 `PETSc.Log.Event` 标注的事件. 主要几列的含义是:

- `Count`: 该事件被调用的次数, 和 `Time`/`Flop` 一样是最大值加最大最小比值;
- `Time (sec)`: 各进程耗时的最大值, 后面跟着最大值与最小值之比;
- `Flop`: 浮点运算数, 同样是最大值加比值;
- `Mess`、`AvgLen`、`Reduct`: MPI 消息条数、平均消息长度 (字节) 和全局归约次数;
- `%T %F %M %L %R`: 该事件占总时间、总浮点运算数、总消息条数、总消息长度、总归约次数的百分比. `--- Global ---` 是相对整个程序, `--- Stage ----` 是相对所在 `Stage`;
- `Mflop/s`: 各进程浮点运算数之和除以最大耗时.

排查性能问题时, 通常先按 `%T` 找出最费时间的几项, 再看它们的 `Mess`/`Reduct` 和最大最小比值, 判断瓶颈是计算本身还是通信与负载不均衡. 并行情形下各列的详细解释见
[Interpreting -log_view Output: Parallel Performance](https://petsc.org/release/manual/profiling/#interpreting-log-view-output-parallel-performance).
::::

::::lang-en
The output of `-log_view` has three parts.

The first part is a global summary giving the wall-clock time `Time (sec)`, the
total number of floating-point operations `Flops`, the number of MPI messages and
the number of global reductions of the whole run. Every row has the four columns
`Max`, `Max/Min`, `Avg` and `Total`: the maximum over the processes, the ratio of
the maximum to the minimum, the average and the sum. A `Max/Min` close to 1 means
the load is balanced across the processes, and the larger the ratio the worse the
balance; in a serial run the ratio of every nonzero quantity is 1 (when a quantity
is itself 0 PETSc simply prints `0.0`), so it carries no information.

The second part, `Summary of Stages`, aggregates by stage. Unless stages are set
up on purpose, everything is recorded in `Main Stage`.

The third part is the event table, one line per event, containing both PETSc's
internal events such as `KSPSolve` and `MatMult` and the events one marks oneself with `PETSc.Log.Event`. The main columns mean:

- `Count`: how many times the event was called; like `Time`/`Flop` it is the
  maximum followed by the max/min ratio;
- `Time (sec)`: the maximum of the elapsed times over the processes, followed by
  the ratio of the maximum to the minimum;
- `Flop`: the number of floating-point operations, again the maximum followed by
  the ratio;
- `Mess`, `AvgLen`, `Reduct`: the number of MPI messages, the average message
  length (in bytes) and the number of global reductions;
- `%T %F %M %L %R`: the percentage of the total time, the total floating-point
  operations, the total number of messages, the total message length and the
  total number of reductions taken by this event. `--- Global ---` is relative to
  the whole program, `--- Stage ----` relative to the stage it belongs to;
- `Mflop/s`: the sum of the floating-point operations over the processes divided
  by the maximum elapsed time.

When tracking down a performance problem one usually first picks out the most
expensive items by `%T`, and then looks at their `Mess`/`Reduct` and max/min
ratios to decide whether the bottleneck is the computation itself or
communication and load imbalance. A detailed explanation of the columns in the
parallel case is given in
[Interpreting -log_view Output: Parallel Performance](https://petsc.org/release/manual/profiling/#interpreting-log-view-output-parallel-performance).
::::

### [火焰图]{.lang-zh} [Flame graphs]{.lang-en}

::::lang-zh
运行脚本时把选项换成 `-log_view :profile.txt:ascii_flamegraph`, PETSc 会把日志写成火焰图格式 (命令里的 `py/profile_poisson.py` 是本章最后一节"写成脚本运行"给出的脚本):
::::

::::lang-en
Replacing the option with `-log_view :profile.txt:ascii_flamegraph` when running a
script makes PETSc write the log in flame graph format (the
`py/profile_poisson.py` in the command is the script given in the last section of
this chapter, "Running as a script"):
::::

```bash
python3 py/profile_poisson.py -log_view :profile.txt:ascii_flamegraph
```

::::lang-zh
生成的 `profile.txt` 每行是一条调用栈和它的耗时 (单位是微秒), 形如
::::

::::lang-en
Every line of the resulting `profile.txt` is a call stack together with its
elapsed time (in microseconds), of the form
::::

```
Main Stage;firedrake;MyAssemble;firedrake.assemble.assemble;ParLoopExecute 288
```

::::lang-zh
把这个文件上传到在线工具 [speedscope](https://www.speedscope.app/) 就能看到火焰图;
也可以用 [FlameGraph](https://github.com/brendangregg/FlameGraph) 提供的
`flamegraph.pl` 生成 SVG, 注意加上 `--countname us`, 否则鼠标悬停时显示的单位会写成
samples 而不是微秒. 火焰图比表格多出来的信息是事件之间的嵌套关系: Firedrake 自身的不少函数也带着事件装饰器, 因此能一眼看出时间是花在组装、代码编译还是求解上.
::::

::::lang-en
Uploading this file to the online tool [speedscope](https://www.speedscope.app/)
shows the flame graph; alternatively `flamegraph.pl` from
[FlameGraph](https://github.com/brendangregg/FlameGraph) generates an SVG; note that `--countname us` should be added, otherwise the unit shown on hover reads samples
instead of microseconds. What a flame graph adds over the table is the nesting of
the events: many of Firedrake's own functions carry event decorators too, so one
can see at a glance whether the time goes into assembly, code compilation or the
solve.
::::

### [自定义事件]{.lang-zh} [Custom events]{.lang-en}

::::lang-zh
`-log_view` 默认只统计 PETSc 与 Firedrake 内部的事件. 用 `PETSc.Log.Event` 和
`PETSc.Log.EventDecorator` 可以把自己关心的代码块也标注成事件, 它的名字和运行时间会一并出现在上面的表格和火焰图中.
::::

::::lang-en
By default `-log_view` only accounts for the events internal to PETSc and
Firedrake. With `PETSc.Log.Event` and `PETSc.Log.EventDecorator` the blocks of
code we care about can be marked as events as well, and their names and run times
then appear in the tables and flame graphs above.
::::

1. [`PETSc.Log.Event` 用作上下文管理器, 适合标注一段代码:]{.lang-zh}[`PETSc.Log.Event` is used as a context manager, suitable for marking a block of code:]{.lang-en}

    ```python
    from firedrake.petsc import PETSc

    with PETSc.Log.Event("foo"):
        do_something_expensive()
    ```

2. [`PETSc.Log.EventDecorator` 用作装饰器, 适合标注整个函数:]{.lang-zh}[`PETSc.Log.EventDecorator` is used as a decorator, suitable for marking a whole function:]{.lang-en}

    ```python
    from firedrake.petsc import PETSc

    @PETSc.Log.EventDecorator("foo")
    def do_something_expensive():
        ...
    ```

    [不传名字时, 事件名取"模块名.函数名"的形式. 事件表里的 `firedrake.assemble.assemble` 就是这么来的.]{.lang-zh}[Without a name, the event name takes the form "module name.function name". This is where `firedrake.assemble.assemble` in the event table comes from.]{.lang-en}

::::lang-zh
事件名会原样出现在事件表的第一列, 而这一列的宽度是 16 个字符, 名字一旦超过 16 个字符, 这一行后面的所有列都会整体右移, 和表头对不齐, 因此事件名要尽量取短.
::::

::::lang-en
The event name appears verbatim in the first column of the event table, and that
column is 16 characters wide; once a name exceeds 16 characters all the remaining
columns of that line are shifted to the right and no longer line up with the
header, so event names should be kept short.
::::

### [自定义 `Stage`]{.lang-zh} [Custom stages]{.lang-en}

::::lang-zh
`Stage` 把程序切成前后相继的几段. `-log_view` 的 `Summary of Stages` 给每个 `Stage` 一行汇总, 事件表也按 `Stage` 分块, 同一事件在各 `Stage` 中分开计数. 与事件不同, `Stage` 之间互不包含, 每段时间只计入当前所处的 `Stage`. `PETSc.Log.Stage` 同样用作上下文管理器, `with` 块之外的部分记在默认的 `Main Stage` 里:
::::

::::lang-en
A stage cuts the program into consecutive pieces. The `Summary of Stages` of
`-log_view` gives one line per stage, the event table is split into blocks by
stage, and the same event is counted separately in each stage. Unlike events,
stages do not contain one another: every stretch of time is counted only in the
current stage. `PETSc.Log.Stage` is likewise used as a context manager, and
everything outside the `with` blocks is recorded in the default `Main Stage`:
::::

```python
from firedrake.petsc import PETSc

with PETSc.Log.Stage("MySolve"):
    solver.solve()
```

## [一个可运行的算例]{.lang-zh} [A runnable example]{.lang-en}

::::lang-zh
给一个小规模 Poisson 问题的网格生成、组装和求解分别加上自定义事件, 再读出各事件的耗时. Notebook 里没有命令行, 要用 `PETSc.Log.begin()` 手动打开日志, 用 `getPerfInfo()` 或 `PETSc.Log.view` 取结果; 火焰图等命令行选项见本节末尾的脚本.
::::

::::lang-en
We attach custom events to the mesh generation, assembly and solve of a small
Poisson problem and read off their elapsed times. A notebook has no command line,
so the log is turned on by hand with `PETSc.Log.begin()` and the results are read
with `getPerfInfo()` or `PETSc.Log.view`; for flame graphs and other command-line
options see the script at the end of this section.
::::

In [ ]:
import petsctools
from firedrake.petsc import PETSc

# 命令行传了 -log_view 时 PETSc 已经自动开启日志, 此时不要再调用 Log.begin():
# 那会额外启动一个扁平日志处理器, 用 ascii_xml / ascii_flamegraph 这类嵌套格式时,
# 程序退出会报 "Trying to end paused event, not allowed".
if 'log_view' not in petsctools.get_commandline_options():
    PETSc.Log.begin()

t_start = PETSc.Log.getTime()   # 记下起点, 后面用来算耗时占比

print('logging active:', PETSc.Log.isActive())

::::lang-zh
`petsctools.get_commandline_options()` 返回 PETSc 从命令行认出的选项名. 这个判断 (取自 Firedrake 文档 [Optimising](https://www.firedrakeproject.org/optimising.html)) 让同一段代码在脚本里带 `-log_view` 运行时跳过 `Log.begin()`: 此时 PETSc 已自行开启日志 (普通格式为扁平日志, `ascii_xml`/`ascii_flamegraph` 为嵌套日志), 再调用会多启动一个扁平日志处理器, 用嵌套格式时退出会报 `Trying to end paused event, not allowed`.

`PETSc.Log.begin()` 必须放在被计时的代码之前, 之前发生的事 (如 `import firedrake`) 不会被记录.
::::

::::lang-en
`petsctools.get_commandline_options()` returns the option names PETSc recognizes on
the command line. This test (taken from the Firedrake documentation,
[Optimising](https://www.firedrakeproject.org/optimising.html)) makes the same code
skip `Log.begin()` when run as a script with `-log_view`: PETSc has then already
turned the log on (a flat log for the plain format, a nested log for
`ascii_xml`/`ascii_flamegraph`), and calling it again starts an additional flat log
handler; with a nested format the program then reports
`Trying to end paused event, not allowed` on exit.

`PETSc.Log.begin()` must come before the code being timed; whatever happens before
it (such as `import firedrake`) is not recorded.
::::

In [ ]:
from firedrake import *

N = 64
params = {'ksp_type': 'cg', 'pc_type': 'jacobi', 'ksp_rtol': 1e-10}

with PETSc.Log.Event('MyMesh'):
    mesh = UnitSquareMesh(N, N)
    V = FunctionSpace(mesh, 'CG', 1)

x, y = SpatialCoordinate(mesh)
u, v = TrialFunction(V), TestFunction(V)
a = inner(grad(u), grad(v))*dx
L = inner(sin(pi*x)*sin(pi*y), v)*dx
bc = DirichletBC(V, 0, 'on_boundary')
u_h = Function(V, name='u_h')

with PETSc.Log.Event('MyAssemble'):
    A = assemble(a, bcs=bc)
    b = assemble(L, bcs=bc)

with PETSc.Log.Event('MySolve'):
    solve(A, u_h, b, solver_parameters=params)

print('number of dofs:', V.dim())

::::lang-zh
组装和求解拆成两步, 两个事件的耗时才分得开; 写成 `solve(a == L, u_h, bcs=bc)` 时组装在求解内部完成, 只剩一个数.

用同一个名字再取 `PETSc.Log.Event(name)` 得到的是同一个事件, 其 `getPerfInfo()` 返回一个字典: 调用次数 `count`、累计耗时 `time` (秒)、浮点运算数 `flops`, 以及并行时才非零的 `numMessages`、`messageLength`、`numReductions`.
::::

::::lang-en
Assembly and solve are split into two steps so that the two events can be told
apart; with `solve(a == L, u_h, bcs=bc)` the assembly happens inside the solve and
only one number is left.

Asking for `PETSc.Log.Event(name)` again with the same name gives the same event,
whose `getPerfInfo()` returns a dictionary: the number of calls `count`, the
accumulated time `time` (in seconds), the number of floating-point operations
`flops`, and `numMessages`, `messageLength`, `numReductions`, which are nonzero only
in parallel.
::::

In [ ]:
def show_events(names):
    """打印各事件的调用次数、累计耗时、耗时占比和浮点运算数."""
    elapsed = PETSc.Log.getTime() - t_start
    print('event'.ljust(14) + 'count'.rjust(6) + 'time (s)'.rjust(10)
          + '%T'.rjust(7) + 'flop'.rjust(12))
    for name in names:
        info = PETSc.Log.Event(name).getPerfInfo()
        count, t, flop = info['count'], info['time'], info['flops']
        print(f'{name:<14}{count:>6}{t:>10.4f}{100*t/elapsed:>6.1f}%{flop:>12.3e}')


show_events(['MyMesh', 'MyAssemble', 'MySolve'])

::::lang-zh
`%T` 是各事件耗时占 `t_start` 以来墙钟时间的比例, 分母含 Notebook 单元之间的空闲时间, 只看相对大小即可.

三个事件的耗时都远大于真正的线性代数运算 (后面表格里 `KSPSolve` 只有几毫秒): Firedrake 把变分形式即时编译成 C 代码 (JIT), 第一次组装和求解包含了代码生成、编译、稀疏结构构建和求解器 setup 等一次性开销. 再做一遍就能看出差别:
::::

::::lang-en
`%T` is the share of each event in the wall-clock time since `t_start`; the
denominator includes the idle time between notebook cells, so only its relative size
is meaningful.

All three events take far longer than the linear algebra itself (`KSPSolve` takes
only a few milliseconds in the table below): Firedrake compiles variational forms
into C code just in time (JIT), so the first assembly and solve include one-off
costs such as code generation, compilation, building the sparsity pattern and
solver setup. Doing it once more shows the difference:
::::

In [ ]:
with PETSc.Log.Event('MyAssemble2'):
    A = assemble(a, bcs=bc)
    b = assemble(L, bcs=bc)

u_h.assign(0)      # 让两次求解从同一个初值出发, 迭代步数才可比
with PETSc.Log.Event('MySolve2'):
    solve(A, u_h, b, solver_parameters=params)

show_events(['MyAssemble', 'MyAssemble2', 'MySolve', 'MySolve2'])

::::lang-zh
两次组装的 `flop` 相同, 第二次耗时却低了一个数量级以上, 求解亦然. 因此给代码计时前应先"热身"一遍, 排除编译和 setup 开销; 反过来, 若程序只求解一次, 编译开销就是真实成本, 应关注缓存复用而非求解器优化. 编译结果缓存在磁盘上, 第二次运行时"第一次组装"也比冷启动快, 但上述对比仍然成立.
::::

::::lang-en
The two assemblies have the same `flop` count, yet the second takes more than an
order of magnitude less time, and likewise for the solves. So before timing code one
should warm up by running it once to exclude compilation and setup costs;
conversely, if a program solves only once, the compilation cost is a real cost, and
the focus should be on reusing the cache rather than on optimizing the solver. The
compiled code is cached on disk, so on a second run even the "first assembly" is
faster than from cold, but the comparison above still holds.
::::

::::lang-zh
下面打印完整的 `-log_view` 表格. `PETSc.Log.view` 不给 viewer 时会连同主机名、用户名一起写到标准输出, 所以这里写入临时文件后只挑需要的行.
::::

::::lang-en
Next we print the full `-log_view` table. Without a viewer `PETSc.Log.view` writes to
standard output together with the host name and user name, so here it is written to
a temporary file and only the required lines are printed.
::::

In [ ]:
import os
import tempfile

with tempfile.TemporaryDirectory() as tmpdir:
    filename = os.path.join(tmpdir, 'profile.txt')
    viewer = PETSc.Viewer().createASCII(filename, comm=COMM_WORLD)
    PETSc.Log.view(viewer)
    viewer.destroy()            # 先关闭再读, 确保内容已经写入文件
    with open(filename) as f:
        lines = f.read().splitlines()

summary = ('Time (sec):', 'Flops:', 'Event  ', '                   Max Ratio')
events = ('MyMesh', 'MyAssemble', 'MyAssemble2', 'MySolve', 'MySolve2',
          'KSPSolve', 'MatMult', 'PCApply', 'firedrake.assemble.assemble')
for line in lines:
    if line.startswith(summary) or line.split(' ')[0] in events:
        print(line)

::::lang-zh
这就是脚本加 `-log_view` 时看到的表格 (只留了表头和几个事件), 可对照"读懂 `-log_view` 的输出"一节: `KSPSolve` 的 `Count` 为 2, 对应两次求解; 单进程时非零量的比值都是 1.0 (量为 0 时打成 `0.0`), `Mess`、`AvgLen`、`Reduct` 全为 0. `firedrake.assemble.assemble` 是 Firedrake 自己的事件, 名字超过 16 个字符, 后面的列整体右移了.
::::

::::lang-en
This is the table a script shows with `-log_view` (only the header and a few events
are kept); compare it with "Reading the output of `-log_view`": the `Count` of
`KSPSolve` is 2, matching the two solves; in a single-process run the ratio of
every nonzero quantity is 1.0 (a quantity that is 0 is printed as `0.0`), and
`Mess`, `AvgLen`, `Reduct` are all 0. `firedrake.assemble.assemble` is Firedrake's
own event; its name exceeds 16 characters, so the columns after it are shifted to
the right.
::::

::::lang-zh
把组装和求解放进 `Stage` `MyResolve` 再做一遍, 然后按 `Stage` 读取事件统计. `getPerfInfo` 的参数 `stage` 取 `Stage` 的编号 `id`, `Main Stage` 为 0.
::::

::::lang-en
The assembly and solve are done once more inside a stage `MyResolve`, and the event
statistics are then read stage by stage. The argument `stage` of `getPerfInfo` takes
the number `id` of the stage; `Main Stage` is 0.
::::

In [ ]:
stage = PETSc.Log.Stage('MyResolve')
with stage:
    A = assemble(a, bcs=bc)
    b = assemble(L, bcs=bc)
    u_h.assign(0)
    solve(A, u_h, b, solver_parameters=params)

main = PETSc.Log.Stage('Main Stage')
print('stage ids:', main.id, stage.id)
print('stage'.ljust(12) + 'event'.ljust(10) + 'count'.rjust(6) + 'time (s)'.rjust(10))
for st in (main, stage):
    for name in ('KSPSolve', 'MatMult'):
        info = PETSc.Log.Event(name).getPerfInfo(stage=st.id)
        print(f"{st.name:<12}{name:<10}{info['count']:>6}{info['time']:>10.4f}")

::::lang-zh
`Main Stage` 里的 `KSPSolve` 是前面的两次求解, `MyResolve` 里只有这一次. 注意不传 `stage` 时 `getPerfInfo()` 读的是**当前** `Stage`, 要读其他 `Stage` 中的事件须显式传 `stage`.
::::

::::lang-en
The `KSPSolve` in `Main Stage` are the two earlier solves, and `MyResolve` has only
this one. Note that without `stage`, `getPerfInfo()` reads the **current** stage;
to read events in another stage, `stage` must be passed explicitly.
::::

In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
    filename = os.path.join(tmpdir, 'profile.txt')
    viewer = PETSc.Viewer().createASCII(filename, comm=COMM_WORLD)
    PETSc.Log.view(viewer)
    viewer.destroy()
    with open(filename) as f:
        lines = f.read().splitlines()

start = next(i for i, line in enumerate(lines) if line.startswith('Summary of Stages'))
print('\n'.join(lines[start:start + 4]))
print()
for line in lines[start:]:
    if line.startswith('Object Type'):      # 之后是对象统计, 不再看
        break
    if line.startswith('--- Event Stage') or line.split(' ')[0] == 'KSPSolve':
        print(line)

::::lang-zh
`Summary of Stages` 多了一行 `MyResolve`, 事件表分成两块. `KSPSolve` 行的前一组百分比相对整个程序, 后一组相对所在 `Stage`; `MyResolve` 很短, 所以它的 `%T` 在后一组里大得多.
::::

::::lang-en
`Summary of Stages` gains a line for `MyResolve`, and the event table splits into two
blocks. On the `KSPSolve` lines the first group of percentages is relative to the
whole program and the second to the stage; `MyResolve` is short, so its `%T` is much
larger in the second group.
::::

### [写成脚本运行]{.lang-zh} [Running as a script]{.lang-en}

::::lang-zh
命令行选项只能用在脚本上. 脚本 [py/profile_poisson.py](py/profile_poisson.py) 是同一个算例, 但去掉了 `PETSc.Log.begin()`, 改由 `-log_view` 自动开启日志, 并且用装饰器的写法标注组装:
::::

::::lang-en
Command-line options can only be used with scripts. The script
[py/profile_poisson.py](py/profile_poisson.py) is the same example, but with
`PETSc.Log.begin()` removed: the log is turned on automatically by `-log_view` instead, and the
assembly is marked with the decorator form:
::::

:::{literalinclude} py/profile_poisson.py
:::

[在激活了 Firedrake 环境的终端中, 切换到本笔记本所在的 `firedrake` 目录后运行]{.lang-zh}[In a terminal with the Firedrake environment activated, change to the `firedrake` directory containing this notebook and run]{.lang-en}

```bash
python3 py/profile_poisson.py -log_view
```

[即可看到完整的性能摘要; 换成]{.lang-zh}[to see the full performance summary; replacing it with]{.lang-en}

```bash
python3 py/profile_poisson.py -log_view :profile.txt:ascii_flamegraph
```

[则会在当前目录生成火焰图文件 `profile.txt`. 用]{.lang-zh}[generates the flame graph file `profile.txt` in the current directory. Running]{.lang-en}

```bash
mpiexec -n 4 python3 py/profile_poisson.py -log_view
```

::::lang-zh
并行运行时, 最大最小比值和 MPI 相关的列才有内容.

找到瓶颈后的改法见 Firedrake 文档 [Optimising](https://www.firedrakeproject.org/optimising.html) 的 Common performance issues 一节. 最常见的一条: 每次 `solve()` 都会新建求解器对象, 循环中反复求解时应在循环外建好 `NonlinearVariationalProblem` 与 `NonlinearVariationalSolver` 并复用.
::::

::::lang-en
in parallel gives the max/min ratios and the MPI-related columns real content.

For fixing a bottleneck see the Common performance issues section of
[Optimising](https://www.firedrakeproject.org/optimising.html) in the Firedrake
documentation. The most common item: every `solve()` creates a new solver object,
so when solving repeatedly in a loop, create the `NonlinearVariationalProblem` and
`NonlinearVariationalSolver` outside the loop and reuse them.
::::

## [并行扩展性]{.lang-zh} [Parallel scalability]{.lang-en}

::::lang-zh
固定问题的总规模, 增加进程数, 看耗时如何下降, 这叫强扩展性 (strong scaling) 测试. 记 $T_p$ 为用 $p$ 个进程时的耗时, 加速比 $S_p = T_1/T_p$, 并行效率 $E_p = S_p/p$; 理想情形下 $S_p = p$, $E_p = 1$. 程序的各个部分扩展性往往不同, 只看总时间说明不了问题, 应当按 `Stage` 分开看, 这正是上一节自定义 `Stage` 的用处.

脚本 [py/scaling_poisson.py](py/scaling_poisson.py) 在 $128\times 128$ 的粗网格上用 `MeshHierarchy` 加密 3 次 (约 105 万个自由度), 以几何多重网格为预条件子用 CG 求解 Poisson 方程, 并把程序分成建网格 (`MyMesh`)、建空间与变分形式 (`MySpace`)、求解 (`MySolve`, 含组装) 三个 `Stage`:
::::

::::lang-en
Fixing the total size of the problem, increasing the number of processes and
watching how the elapsed time goes down is called a strong scaling test. Write
$T_p$ for the elapsed time with $p$ processes; the speedup is $S_p = T_1/T_p$ and
the parallel efficiency $E_p = S_p/p$; ideally $S_p = p$ and $E_p = 1$. The parts
of a program often scale differently, so the total time alone says little; one
should look at the stages separately, which is exactly what the custom stages of
the previous section are for.

The script [py/scaling_poisson.py](py/scaling_poisson.py) refines a $128\times 128$
coarse mesh 3 times with `MeshHierarchy` (about 1.05 million degrees of freedom),
solves the Poisson equation with CG preconditioned by geometric multigrid, and
divides the program into three stages: mesh generation (`MyMesh`), function space
and variational form (`MySpace`), and the solve (`MySolve`, including assembly):
::::

:::{literalinclude} py/scaling_poisson.py
:::

::::lang-zh
测试前先用小规模把每种进程数都跑一遍, 预热 Firedrake 的编译缓存, 免得第一次运行的编译时间混进测量. 然后依次用 1、2、4、8 个进程运行, 每次把日志写成一个 CSV 文件:
::::

::::lang-en
Before the test, every process count is run once at a small size to warm
Firedrake's compilation cache, so that the compilation time of a first run does not
end up in the measurement. Then the script is run with 1, 2, 4 and 8 processes in
turn, each run writing its log to a CSV file:
::::

```bash
for p in 1 2 4 8; do mpiexec -n $p python3 py/scaling_poisson.py -N 16 -refine 1; done
for p in 1 2 4 8; do
    mpiexec -n $p python3 py/scaling_poisson.py -log_view :poisson-np$p.csv:ascii_csv
done
```

::::lang-zh
`ascii_csv` 是 `-log_view` 的又一种输出格式, 便于用程序读取, 不必从文本表格里截取列. 文件的每一行依次是 `Stage` 名、事件名、进程号、调用次数、耗时、消息条数、消息长度、归约次数和浮点运算数, 每个进程各占一行; 每个 `Stage` 还有一行事件名为 `summary` 的记录, 给出该 `Stage` 的总耗时. 表头末尾还有 `dof0`、`e0` 等几列, 这里用不到, 数据行里也没有.

下面的数据是在一台 8 个性能核的 Apple M2 Max 笔记本上用 Firedrake 2026.4.1 测得的. 仓库里的 `assets/scaling/poisson-np*.csv` 只保留了 `summary` 和几个关心的事件, 以减小文件体积; 读取的代码对完整的文件同样适用. 换一台机器, 绝对数值和扩展性都会不同.
::::

::::lang-en
`ascii_csv` is yet another output format of `-log_view`; it is convenient to read
from a program, with no need to cut columns out of a text table. Every line of the
file holds, in order, the stage name, the event name, the rank, the number of calls,
the elapsed time, the number of messages, the message length, the number of
reductions and the number of floating-point operations, one line per process;
every stage also has a line whose event name is `summary`, giving the total time of
that stage. The header ends with a few more columns such as `dof0` and `e0`, which
are not needed here and are absent from the data lines.

The data below were measured with Firedrake 2026.4.1 on an Apple M2 Max laptop with
8 performance cores. The files `assets/scaling/poisson-np*.csv` in the
repository keep only `summary` and a few events of interest, to reduce their size;
the reading code works equally well on the full files. On another machine both the
absolute numbers and the scalability will differ.
::::

In [ ]:
import csv
from collections import defaultdict


def read_log_csv(filename):
    """读取 -log_view :file:ascii_csv 的输出, 返回 {(Stage, 事件): 各进程耗时的最大值}."""
    times = defaultdict(float)
    with open(filename) as f:
        reader = csv.reader(f)
        next(reader)                          # 跳过表头
        for row in reader:
            stage, event, rank, count, time = row[:5]
            if int(count) > 0:
                times[stage, event] = max(times[stage, event], float(time))
    return times


nprocs = [1, 2, 4, 8]
logs = {p: read_log_csv(f'assets/scaling/poisson-np{p}.csv') for p in nprocs}

items = [('MyMesh', 'summary'), ('MySolve', 'summary'), ('MySolve', 'PCSetUp'),
         ('MySolve', 'KSPSolve'), ('Main Stage', 'summary')]
print('stage/event'.ljust(20) + ''.join(f'np={p}'.rjust(8) for p in nprocs) + 'speedup'.rjust(9))
for stage, event in items:
    t = [logs[p][stage, event] for p in nprocs]
    label = stage if event == 'summary' else f'{stage}/{event}'
    print(f'{label:<20}' + ''.join(f'{ti:>8.2f}' for ti in t) + f'{t[0]/t[-1]:>9.1f}')

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5, 3.5))
for stage, event in items[:4]:
    t = [logs[p][stage, event] for p in nprocs]
    label = stage if event == 'summary' else f'{stage}/{event}'
    ax.loglog(nprocs, t, 'o-', label=label)
    ax.loglog(nprocs, [t[0]/p for p in nprocs], 'k:', lw=0.8)   # 理想扩展
ax.set_xticks(nprocs, [str(p) for p in nprocs])
ax.xaxis.set_minor_locator(plt.NullLocator())   # 去掉对数轴上的次刻度标签
ax.set_xlabel('number of processes')
ax.set_ylabel('time (s)')
ax.legend(fontsize=8)
plt.show()

::::lang-zh
图中虚线是从单进程出发的理想扩展. `MySpace` 的耗时不到 1 毫秒, 表中略去. 按 `Stage` 看:

- **`MyMesh`** 是这个算例里最贵的部分, 比求解还贵: 构建网格层次时要逐层加密, 并给每一层建立坐标空间与编号. 这些基本是逐单元的局部操作, 扩展性最好, 8 个进程时加速比约 5.6.
- **`MySolve`** 的大部分时间在 `PCSetUp`, 即多重网格预条件子的 setup: 为各层建立粗网格上的问题, 其中包括层间的限制与注入. 它和 `MyMesh` 一样是一次性开销, 加速比约 4.9.
- **`KSPSolve`**, 也就是真正的 Krylov 迭代, 只占不到 1 秒, 加速比却只有 2.2. CG 的迭代次数基本不变 (1 个进程时 4 次, 其余 5 次; 并行时松弛只在各进程内部做, 预条件效果略有变化), 所以这里反映的是每次迭代的开销. 稀疏矩阵向量乘和松弛这类运算通常受内存带宽限制, 同一台机器上的各个核共享内存带宽, 进程越多每个进程分到的越少; 粗层上每个进程分到的自由度又很少, 通信与同步的比重随之变大.
- **`Main Stage`**, 即不在三个自定义 `Stage` 里的部分, 一直在 2.3 到 2.7 秒之间, 不随进程数下降. 从完整日志看, 其中主要是 `import firedrake` 的后半段和构造 `LinearVariationalSolver` (含为矩阵分配稀疏结构). 进程越多, 这类不能并行的部分在总时间里占比越大, 这就是 Amdahl 定律说的情形.

由此可见, 只看总时间会得出"8 个进程快了约 4.6 倍"的笼统结论, 而按 `Stage` 看就知道: 若这个程序要在同一网格上反复求解, 真正重复发生的 `KSPSolve` 恰恰是扩展性最差的部分, 优化的重点应当放在那里.
::::

::::lang-en
The dotted lines in the figure are ideal scaling starting from the single-process
time. `MySpace` takes less than 1 millisecond and is left out of the table. Stage by
stage:

- **`MyMesh`** is the most expensive part of this example, more expensive than the
  solve: building the mesh hierarchy refines level by level and sets up a
  coordinate space and a numbering for every level. These are essentially local,
  cell-by-cell operations, which scale best, with a speedup of about 5.6 on 8
  processes.
- **`MySolve`** spends most of its time in `PCSetUp`, the setup of the multigrid
  preconditioner: building the problems on the coarse levels, including the
  restriction and injection between levels. Like `MyMesh` it is a one-off cost,
  with a speedup of about 4.9.
- **`KSPSolve`**, the actual Krylov iteration, takes less than 1 second, but its
  speedup is only 2.2. The number of CG iterations hardly changes (4 on 1 process,
  5 otherwise; in parallel the relaxation is done only within each process, so the
  preconditioner behaves slightly differently), so what is seen here is the cost per
  iteration. Operations such as sparse matrix-vector products and relaxation are
  usually limited by memory bandwidth; the cores of one machine share the memory
  bandwidth, so the more processes there are, the less each one gets; moreover each
  process owns very few degrees of freedom on the coarse levels, so communication
  and synchronization take a growing share.
- **`Main Stage`**, the part outside the three custom stages, stays between 2.3 and
  2.7 seconds and does not go down with the number of processes. The full log shows
  that it consists mainly of the second half of `import firedrake` and the
  construction of the `LinearVariationalSolver` (including allocating the sparsity
  pattern of the matrix). The more processes there are, the larger the share of
  such non-parallelizable parts in the total time, which is the situation
  described by Amdahl's law.

So the total time alone leads to the vague conclusion that "8 processes are about
4.6 times faster", whereas looking at the stages shows that if this program were to
solve repeatedly on the same mesh, `KSPSolve`, the part that actually repeats, is
precisely the part that scales worst, and that is where optimization should focus.
::::